# Aula 17 — Preparação de dados para Machine Learning

**Módulo 6 — Introdução ao Machine Learning**

## Objetivos da aula

- Entender a separação entre atributos de entrada (*features*) e variável-alvo (*target*).
- Selecionar as variáveis que serão usadas pelo modelo.
- Organizar os dados para uma tarefa de aprendizado supervisionado.

---

## 1. O que é aprendizado supervisionado

Em **aprendizado supervisionado**, treinamos um modelo mostrando a ele muitos exemplos de entradas **já com a resposta certa**, para que ele aprenda o padrão e consiga prever a resposta em novos casos, nunca vistos antes. No nosso cenário: mostramos ao modelo temperatura, pressão, vibração etc. de vários equipamentos, **junto com o status real** de cada um (`normal`, `alerta`, `critico`), até que ele aprenda a associar os padrões de leitura a cada status.

Essa é exatamente a tarefa que vamos construir nesta e nas próximas duas aulas — o fechamento do curso.

Todo problema de aprendizado supervisionado é organizado em dois blocos:

- **`X` (atributos / *features*)**: as variáveis de entrada, usadas para prever.
- **`y` (variável-alvo / *target*)**: o que queremos prever.

## 2. Recriando a base de dados

In [7]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 500
tipos_equipamento = ["Motor", "Bomba", "Compressor", "Ventilador"]

dados = pd.DataFrame({
    "equipamento_id": [f"EQ-{i:04d}" for i in range(1, n + 1)],
    "tipo_equipamento": np.random.choice(tipos_equipamento, size=n),
    "temperatura": np.round(np.random.normal(70, 12, size=n), 1),
    "pressao": np.round(np.random.normal(5.5, 1.3, size=n), 2),
    "vibracao": np.round(np.random.normal(2.4, 1.1, size=n), 2),
    "horas_operacao": np.random.randint(0, 10000, size=n),
})


def definir_status(linha):
    critico = (linha["temperatura"] >= 90) or (linha["vibracao"] >= 4.5) or (linha["pressao"] >= 8) or (linha["pressao"] <= 2)
    alerta = (linha["temperatura"] >= 80) or (linha["vibracao"] >= 3.5) or (linha["pressao"] >= 7) or (linha["pressao"] <= 3)
    if critico:
        return "critico"
    elif alerta:
        return "alerta"
    else:
        return "normal"


dados["status"] = dados.apply(definir_status, axis=1)
dados.to_csv("sensores_industriais.csv", index=False)

df = pd.read_csv("sensores_industriais.csv")
df.head()


,equipamento_id,tipo_equipamento,temperatura,pressao,vibracao,horas_operacao,status
0,EQ-0001,Compressor,59.8,6.42,3.01,1958,normal
1,EQ-0002,Ventilador,51.8,6.08,1.33,6344,normal
2,EQ-0003,Motor,64.6,5.03,2.52,5779,normal
3,EQ-0004,Compressor,80.3,7.01,0.93,6144,alerta
4,EQ-0005,Compressor,72.6,4.09,1.74,5063,normal


## 3. Definindo a variável-alvo (`y`)

Queremos prever o `status` do equipamento — essa é a nossa variável-alvo.

In [8]:
y = df["status"]

print(y.value_counts())


status
normal     280
alerta     161
critico     59
Name: count, dtype: int64


Repare que as três classes não têm o mesmo número de exemplos — isso é chamado de **desbalanceamento de classes** e será importante na próxima aula, na hora de dividir os dados entre treino e teste.

## 4. Selecionando os atributos (`X`)

Nem toda coluna da base deve entrar no modelo. `equipamento_id`, por exemplo, é apenas um identificador — não carrega nenhuma informação sobre o comportamento físico do equipamento, e usá-lo faria o modelo "decorar" identificadores em vez de aprender um padrão real. Já `temperatura`, `pressao`, `vibracao` e `horas_operacao` são medidas diretamente relacionadas ao estado do equipamento: são boas candidatas a atributos.

In [9]:
atributos_numericos = ["temperatura", "pressao", "vibracao", "horas_operacao"]

X_numerico = df[atributos_numericos]
X_numerico.head()


,temperatura,pressao,vibracao,horas_operacao
0,59.8,6.42,3.01,1958
1,51.8,6.08,1.33,6344
2,64.6,5.03,2.52,5779
3,80.3,7.01,0.93,6144
4,72.6,4.09,1.74,5063


## 5. Codificando variáveis categóricas

A coluna `tipo_equipamento` também pode ser útil (equipamentos diferentes têm padrões diferentes), mas é **texto**, e os algoritmos de Machine Learning do Scikit-learn trabalham apenas com números. Para incluí-la, usamos o **one-hot encoding**: criar uma coluna binária (0/1) para cada categoria possível, com `pd.get_dummies()`.

In [10]:
tipo_codificado = pd.get_dummies(df["tipo_equipamento"], prefix="tipo", dtype=int)
tipo_codificado.head()


,tipo_Bomba,tipo_Compressor,tipo_Motor,tipo_Ventilador
0,0,1,0,0
1,0,0,0,1
2,0,0,1,0
3,0,1,0,0
4,0,1,0,0


Cada linha tem exatamente um `1`, na coluna correspondente ao seu tipo real de equipamento, e `0` nas demais — assim a informação categórica é representada numericamente, sem sugerir uma ordem que não existe entre as categorias (o que aconteceria se simplesmente numerássemos `Motor=1, Bomba=2, ...`).

## 6. Montando a tabela final de atributos (`X`)

Juntamos os atributos numéricos com as colunas codificadas, usando `pd.concat()` (a mesma função de "juntar tabelas" que vimos na Aula 14, ao unir linhas — aqui usamos `axis=1` para unir **colunas**).

In [11]:
X = pd.concat([X_numerico, tipo_codificado], axis=1)

print("Formato de X (atributos):", X.shape)
print("Formato de y (alvo):     ", y.shape)
X.head()


Formato de X (atributos): (500, 8)
Formato de y (alvo):      (500,)


,temperatura,pressao,vibracao,horas_operacao,tipo_Bomba,tipo_Compressor,tipo_Motor,tipo_Ventilador
0,59.8,6.42,3.01,1958,0,1,0,0
1,51.8,6.08,1.33,6344,0,0,0,1
2,64.6,5.03,2.52,5779,0,0,1,0
3,80.3,7.01,0.93,6144,0,1,0,0
4,72.6,4.09,1.74,5063,0,1,0,0


## 7. Conferindo a consistência entre `X` e `y`

Antes de seguir adiante, é sempre importante confirmar que `X` e `y` têm o **mesmo número de linhas** — cada linha de `X` precisa corresponder à linha de `y` na mesma posição, senão o modelo aprenderia associações erradas.

In [12]:
assert len(X) == len(y), "X e y precisam ter o mesmo número de linhas!"
print("Tudo certo: X e y estão alinhados, com", len(X), "registros cada.")


Tudo certo: X e y estão alinhados, com 500 registros cada.


## 8. Resumo da aula

- Aprendizado supervisionado separa os dados em `X` (atributos) e `y` (alvo, o que queremos prever).
- Nem toda coluna deve virar atributo — identificadores como `equipamento_id` não carregam padrão útil.
- Variáveis categóricas precisam ser convertidas em números; `pd.get_dummies()` faz o one-hot encoding.
- `X` e `y` devem sempre ter o mesmo número de linhas, alinhadas na mesma ordem.

### Exercício sugerido

Verifique quantas colunas `X` tem ao todo (`X.shape[1]`) e liste seus nomes com `list(X.columns)`. Você consegue identificar quais vieram dos atributos numéricos e quais vieram do one-hot encoding?
